# Chatbot tra cứu Nghị định: KB + RAG

Một notebook gồm các bước đọc PDF, tạo Knowledge Base, thử RAG và tạo app web responsive. PDF phải nằm trong thư mục `data/`.


## 1. Cài thư viện

In [1]:
import os
import shutil
import pytesseract

candidates = [
    shutil.which("tesseract"),
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
]

tesseract_path = next(
    (p for p in candidates if p and os.path.isfile(p)),
    None
)

print("Tesseract path:", tesseract_path)

if tesseract_path is None:
    raise FileNotFoundError(
        "Không tìm thấy tesseract.exe. Hãy kiểm tra thư mục cài đặt Tesseract."
    )

pytesseract.pytesseract.tesseract_cmd = tesseract_path
print(pytesseract.get_tesseract_version())

Tesseract path: C:\Program Files\Tesseract-OCR\tesseract.exe
5.5.3.20260724


## 2. Tạo chỉ mục KB
TF-IDF là RAG rất nhẹ, chạy CPU, không cần tải model embedding.

In [2]:
from pathlib import Path
from io import BytesIO
import re, joblib, pdfplumber, fitz, pytesseract
import shutil
_tess = shutil.which('tesseract') or r'C:\Program Files\Tesseract-OCR\tesseract.exe'
if Path(_tess).exists(): pytesseract.pytesseract.tesseract_cmd = _tess
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
ROOT=Path.cwd(); PDF_DIR=ROOT/'data'; OUT=ROOT/'storage'/'rag_index.joblib'
def cut(t,size=900,overlap=140):
 t=re.sub(r'\s+',' ',t or '').strip(); r=[]; p=0
 while p<len(t):
  e=min(p+size,len(t)); k=max(t.rfind('. ',p,e),t.rfind('; ',p,e)) if e<len(t) else -1
  if k>p+size//2:e=k+1
  if len(t[p:e].strip())>80:r.append(t[p:e].strip())
  if e>=len(t):break
  p=max(e-overlap,p+1)
 return r
def ocr(f,n,text):
 if text.strip():return text
 doc=fitz.open(f); pix=doc[n-1].get_pixmap(matrix=fitz.Matrix(2,2));doc.close()
 return pytesseract.image_to_string(Image.open(BytesIO(pix.tobytes('png'))),lang='vie+eng')
def build():
 rows=[]
 for f in sorted(PDF_DIR.glob('*.pdf')):
  with pdfplumber.open(f) as pdf:
   for page,pageobj in enumerate(pdf.pages,1):
    rows += [{'text':x,'source':f.name,'page':page} for x in cut(ocr(f,page,pageobj.extract_text() or ''))]
 if not rows:raise ValueError('Không đọc được text PDF.')
 v=TfidfVectorizer(lowercase=True,ngram_range=(1,2),sublinear_tf=True,max_df=.98)
 d={'vectorizer':v,'matrix':v.fit_transform([x['text'] for x in rows]),'records':rows}; OUT.parent.mkdir(exist_ok=True);joblib.dump(d,OUT);return d
kb=build(); print(f'Đã lập chỉ mục {len(kb["records"])} đoạn từ {len(list(PDF_DIR.glob("*.pdf")))} PDF.')

Đã lập chỉ mục 563 đoạn từ 3 PDF.


## 3. Thử truy vấn RAG
Mỗi kết quả luôn mang tên PDF và số trang để đối chiếu.

In [3]:
def search(question,n=4):
 score=cosine_similarity(kb['vectorizer'].transform([question]),kb['matrix']).ravel()
 return [{**kb['records'][i],'score':round(float(score[i]),3)} for i in score.argsort()[::-1][:n] if score[i]>0]
for x in search('Mức phạt không đội mũ bảo hiểm là bao nhiêu?'):
 print(f"{x['source']} | trang {x['page']} | {x['score']}\n{x['text'][:300]}\n")

168-nd-cp.signed.pdf | trang 28 | 0.325
igam/100 mililít máu hoặc vượt quá 0,4 miligam/] lít khí thở; c) Không chấp hành yêu cầu kiểm tra về nồng độ cồn của người thi hành công vụ; d) Người điều khiển xe đạp máy không đội “mũ bảo hiểm cho người đi mô tô, xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy các

168-nd-cp.signed.pdf | trang 17 | 0.279
xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy cách khi điều khiển xe tham gia giao thông trên đường bộ; i) Cho người ngồi trên xe không đội “mũ bảo hiểm cho người đi mô tô, xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy cách, trừ

168-nd-cp.signed.pdf | trang 29 | 0.238
29 đ) Chở người ngồi trên xe đạp máy không đội “mũ bảo hiểm cho người đi mô tô, xe máy” hoặc đội “mũ bảo hiểm cho người đi mô tô, xe máy” không cài quai đúng quy cách, trừ trường hợp chở người bệnh đi cấp cứu, trẻ em dưới 06 tuổi, áp giải người có hành vi vi phạm pháp luật. 5

## 4. Tạo web app
Cell này xuất `app.py` độc lập, gồm toàn bộ RAG, chat UI, lưu SQLite và LLM Groq tùy chọn. Không có `GROQ_API_KEY` app vẫn chạy, chỉ đưa trích đoạn thay vì sinh câu trả lời.

In [4]:
app = r'''
from pathlib import Path
from datetime import datetime
import os,re,sqlite3,joblib,pdfplumber,requests,streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
R=Path(__file__).parent; P=R/'data'; S=R/'storage'; I=S/'rag_index.joblib'; D=S/'history.db'
def pieces(t):
 t=re.sub(r'\s+',' ',t or '').strip();return [t[i:i+900] for i in range(0,len(t),760) if len(t[i:i+900])>80]
def index():
 a=[]
 for f in P.glob('*.pdf'):
  with pdfplumber.open(f) as pdf:
   for p,o in enumerate(pdf.pages,1):a += [{'text':x,'file':f.name,'page':p} for x in pieces(o.extract_text())]
 v=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True);z={'v':v,'m':v.fit_transform([x['text'] for x in a]),'a':a};S.mkdir(exist_ok=True);joblib.dump(z,I);return z
@st.cache_resource
def load():
 z=joblib.load(I) if I.exists() else index()
 if 'v' not in z:z={'v':z['vectorizer'],'m':z['matrix'],'a':[{**r,'file':r.get('source',r.get('file',''))} for r in z['records']]}
 return z
def get(q):
 z=load();s=cosine_similarity(z['v'].transform([q]),z['m']).ravel();return [{**z['a'][i],'score':s[i]} for i in s.argsort()[::-1][:4] if s[i]>0]
def ask(q,c):
 key=os.getenv('GROQ_API_KEY')
 if not key:return None
 ev='\n'.join(f'[{i+1}] {x["text"]}' for i,x in enumerate(c))
 msg=f'Chỉ trả lời bằng tiếng Việt từ tư liệu sau, không suy đoán. Nêu nguồn [1], [2].\nCâu hỏi: {q}\nTư liệu:\n{ev}'
 try:
  x=requests.post('https://api.groq.com/openai/v1/chat/completions',headers={'Authorization':f'Bearer {key}'},json={'model':os.getenv('GROQ_MODEL','llama-3.3-70b-versatile'),'temperature':.1,'messages':[{'role':'user','content':msg}]},timeout=40);x.raise_for_status();return x.json()['choices'][0]['message']['content']
 except requests.RequestException:return None
S.mkdir(exist_ok=True)
with sqlite3.connect(D) as x:x.execute('CREATE TABLE IF NOT EXISTS chats(id INTEGER PRIMARY KEY,role TEXT,text TEXT,at TEXT)')
st.set_page_config('Trợ lý Nghị định','⚖️',layout='centered')
if 'chat' not in st.session_state:st.session_state.chat=[]
st.markdown('<style>.block-container{max-width:880px}@media(max-width:600px){.block-container{padding:1rem .7rem}}</style>',unsafe_allow_html=True)
st.title('⚖️ Trợ lý tra cứu Nghị định');st.caption('Tra cứu mức phạt, điều khoản và nội dung từ PDF đã nạp.')
with st.sidebar:
 if st.button('Lập lại chỉ mục'):index();load.clear();st.rerun()
 if st.button('Xóa hội thoại'):st.session_state.chat=[];st.rerun()
for m in st.session_state.chat:
 with st.chat_message(m['role']):st.markdown(m['text']);st.caption(m.get('src',''))
if q:=st.chat_input('Ví dụ: Không đội mũ bảo hiểm phạt bao nhiêu?'):
 st.session_state.chat.append({'role':'user','text':q})
 with st.chat_message('user'):st.write(q)
 with st.chat_message('assistant'):
  c=get(q);a=ask(q,c) or ('Không tìm thấy đoạn phù hợp.' if not c else 'Các trích đoạn liên quan nhất:\n\n'+'\n\n'.join(f'**[{i+1}]** {x["text"]}' for i,x in enumerate(c)))
  st.markdown(a);src=' • '.join(dict.fromkeys(f'{x["file"]} — trang {x["page"]}' for x in c));st.caption('Nguồn: '+src)
 st.session_state.chat.append({'role':'assistant','text':a,'src':src})
 with sqlite3.connect(D) as x:x.execute('INSERT INTO chats(role,text,at) VALUES(?,?,?)',('user',q,datetime.now().isoformat()));x.execute('INSERT INTO chats(role,text,at) VALUES(?,?,?)',('assistant',a,datetime.now().isoformat()))
'''
Path('app.py').write_text(app,encoding='utf-8');print('Đã tạo app.py')

Đã tạo app.py


## 5. Chạy & deploy Render
Chạy: `streamlit run app.py`, sau đó mở `http://localhost:8501`.

Deploy Render: đưa `app.py`, `requirements.txt`, `render.yaml` và `data/` lên GitHub. Tại Render tạo Web Service; Start Command: `streamlit run app.py --server.port $PORT --server.address 0.0.0.0`. Thêm `GROQ_API_KEY` (tùy chọn) trong Environment Variables. UI tự co giãn cho desktop/mobile.

Lưu ý: SQLite nằm trong `storage/history.db`; filesystem của Render free là tạm thời nên lịch sử có thể mất lúc deploy/restart. Muốn lưu vĩnh viễn, thay SQLite bằng Supabase/Postgres.